# Bybit Data Analysis — Normalised Returns

**Assets:** BTC, ETH, XAUT (Bybit Spot) · S&P 500 (Bloomberg)  
**Granularity:** Hourly  
**Charts:**
1. YTD normalised % returns (since 1 Jan 2026)
2. Normalised % returns since US–Iran conflict began
3. 12-day war (Jun 2025) — uses Yahoo Finance for S&P 500 (Bloomberg data not available)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import nbformat
import requests
import time
import yfinance as yf
from datetime import datetime, timezone

import plotly.io._renderers as _pr
_pr.nbformat = nbformat

from plotting_utils import DEFAULT_THEME, update_timeseries_layout, add_watermark

: 

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CONFIG — adjust dates here
# ══════════════════════════════════════════════════════════════════════
YTD_START         = "2026-01-01"
CONFLICT_START    = "2026-02-28"   # ← set to the date the US-Iran conflict began
END_DATE          = None            # None = now

BYBIT_BASE        = "https://api.bybit.com"
INTERVAL          = "60"           # hourly

# Bybit spot symbols → display names (used in legends)
BYBIT_SYMBOLS = {
    "BTCUSDT":  "Bybit BTC/USDT Spot",
    "ETHUSDT":  "Bybit ETH/USDT Spot",
    "XAUTUSDT": "Bybit XAUT/USDT Spot",
}

# Bloomberg S&P 500 (for YTD and conflict charts)
BBG_SPX_FILE  = r"data\bybit_thahbib_req.xlsx"
BBG_SPX_LABEL = "Bloomberg S&P 500"

# Yahoo Finance S&P 500 (only for June 2025 chart — Bloomberg doesn't go back that far)
YAHOO_SYMBOL = "^GSPC"
YAHOO_LABEL  = "Yahoo Finance S&P 500"

# Colour palette
COLOURS = {
    "Bybit BTC/USDT Spot":   "#ff7f0e",
    "Bybit ETH/USDT Spot":   "purple",
    "Bybit XAUT/USDT Spot":  "#FFD700",
    "Bloomberg S&P 500":     "#E63946",
    "Yahoo Finance S&P 500": "#E63946",
}

In [5]:
# ══════════════════════════════════════════════════════════════════════
# Bybit V5 kline fetcher — paginated hourly data
# ══════════════════════════════════════════════════════════════════════
def fetch_bybit_klines(
    symbol: str,
    interval: str = "60",
    start: str = "2026-01-01",
    end: str | None = None,
    category: str = "spot",
) -> pd.DataFrame:
    """
    Fetch ALL hourly klines between `start` and `end` from Bybit V5,
    paginating backward in 1000-row chunks.
    """
    start_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    end_ms   = int(pd.Timestamp(end, tz="UTC").timestamp() * 1000) if end else int(time.time() * 1000)

    url = f"{BYBIT_BASE}/v5/market/kline"
    all_rows = []
    cursor_end = end_ms

    while cursor_end > start_ms:
        params = {
            "category": category,
            "symbol":   symbol,
            "interval":  interval,
            "start":    start_ms,
            "end":      cursor_end,
            "limit":    1000,
        }
        r = requests.get(url, params=params, timeout=15).json()
        rows = r.get("result", {}).get("list", [])
        if not rows:
            break
        all_rows.extend(rows)
        oldest_ts = int(rows[-1][0])   # list is newest-first
        if oldest_ts <= start_ms:
            break
        cursor_end = oldest_ts - 1
        time.sleep(0.12)               # respect rate limits

    df = pd.DataFrame(all_rows, columns=[
        "open_time", "open", "high", "low", "close", "volume", "turnover",
    ])
    df["open_time"] = pd.to_datetime(df["open_time"].astype(int), unit="ms", utc=True)
    df = df.set_index("open_time").sort_index()
    df["close"] = df["close"].astype(float)
    df = df[~df.index.duplicated(keep="first")]
    return df[["close"]]

In [6]:
# ══════════════════════════════════════════════════════════════════════
# Fetch Bybit data
# ══════════════════════════════════════════════════════════════════════
earliest_start = min(YTD_START, CONFLICT_START)

bybit_frames = {}
for sym, label in BYBIT_SYMBOLS.items():
    print(f"Fetching {sym} from Bybit …", end=" ")
    kdf = fetch_bybit_klines(sym, interval=INTERVAL, start=earliest_start, end=END_DATE)
    kdf = kdf.rename(columns={"close": label})
    bybit_frames[label] = kdf[label]
    print(f"{len(kdf)} rows, {kdf.index.min()} → {kdf.index.max()}")

df_bybit = pd.DataFrame(bybit_frames).sort_index()
print(f"\nBybit combined: {len(df_bybit)} rows")
df_bybit.tail(3)

Fetching BTCUSDT from Bybit … 1697 rows, 2026-01-01 00:00:00+00:00 → 2026-03-12 16:00:00+00:00
Fetching ETHUSDT from Bybit … 1697 rows, 2026-01-01 00:00:00+00:00 → 2026-03-12 16:00:00+00:00
Fetching XAUTUSDT from Bybit … 1697 rows, 2026-01-01 00:00:00+00:00 → 2026-03-12 16:00:00+00:00

Bybit combined: 1697 rows


,Bybit BTC/USDT Spot,Bybit ETH/USDT Spot,Bybit XAUT/USDT Spot
open_time,,,
2026-03-12 14:00:00+00:00,69582.6,2045.22,5109.2
2026-03-12 15:00:00+00:00,70325.8,2072.30,5098.7
2026-03-12 16:00:00+00:00,70426.4,2070.40,5101.0


In [7]:
# ══════════════════════════════════════════════════════════════════════
# Load S&P 500 from Bloomberg Excel (hourly)
# ══════════════════════════════════════════════════════════════════════
print(f"Loading Bloomberg SPX from {BBG_SPX_FILE} …")
_bbg = pd.read_excel(BBG_SPX_FILE, sheet_name="Sheet1")
_bbg["Date"] = pd.to_datetime(_bbg["Date"], utc=True)
_bbg = _bbg.set_index("Date").sort_index()

df_spx = _bbg[["SPX Index  (L1)"]].rename(columns={"SPX Index  (L1)": BBG_SPX_LABEL})
df_spx = df_spx[~df_spx.index.duplicated(keep="first")]
df_spx = df_spx.dropna()

print(f"{len(df_spx)} rows, {df_spx.index.min()} → {df_spx.index.max()}")
df_spx.tail(3)

Loading Bloomberg SPX from data\bybit_thahbib_req.xlsx …
490 rows, 2025-12-10 09:00:00+00:00 → 2026-03-10 16:00:00+00:00


,Bloomberg S&P 500
Date,
2026-03-10 14:00:00+00:00,6796.97
2026-03-10 15:00:00+00:00,6781.55
2026-03-10 16:00:00+00:00,6781.48


In [8]:
# ══════════════════════════════════════════════════════════════════════
# Merge into a single DataFrame (Bloomberg SPX for YTD / conflict)
# ══════════════════════════════════════════════════════════════════════

# Resample Bloomberg SPX to hourly (:00) to align with Bybit's 24/7 grid
df_spx_h = df_spx.resample("1h").last().dropna()

df_all = df_bybit.join(df_spx_h, how="left").sort_index()

# Forward-fill S&P 500 across weekends / overnight (market is closed)
df_all[BBG_SPX_LABEL] = df_all[BBG_SPX_LABEL].ffill()

assets = list(BYBIT_SYMBOLS.values()) + [BBG_SPX_LABEL]
print(f"Merged DataFrame: {len(df_all)} rows · columns: {assets}")
print(f"Range: {df_all.index.min()} → {df_all.index.max()}")
df_all[assets].dropna().tail(5)

Merged DataFrame: 1697 rows · columns: ['Bybit BTC/USDT Spot', 'Bybit ETH/USDT Spot', 'Bybit XAUT/USDT Spot', 'Bloomberg S&P 500']
Range: 2026-01-01 00:00:00+00:00 → 2026-03-12 16:00:00+00:00


,Bybit BTC/USDT Spot,Bybit ETH/USDT Spot,Bybit XAUT/USDT Spot,Bloomberg S&P 500
open_time,,,,
2026-03-12 12:00:00+00:00,70386.0,2065.65,5140.1,6781.48
2026-03-12 13:00:00+00:00,69972.8,2065.46,5114.9,6781.48
2026-03-12 14:00:00+00:00,69582.6,2045.22,5109.2,6781.48
2026-03-12 15:00:00+00:00,70325.8,2072.30,5098.7,6781.48
2026-03-12 16:00:00+00:00,70426.4,2070.40,5101.0,6781.48


In [9]:
# ══════════════════════════════════════════════════════════════════════
# Helper: normalised % return chart
# ══════════════════════════════════════════════════════════════════════
def normalised_return_chart(
    data: pd.DataFrame,
    assets: list[str],
    start: str,
    title_suffix: str = "",
    event_date: str | None = None,
    event_label: str | None = None,
) -> go.Figure:
    """Normalised % returns from `start`, in Block Scholes report style."""
    sliced = data[assets].loc[start:].dropna(how="all")
    base   = sliced.bfill().iloc[0]
    pct    = (sliced / base - 1) * 100

    fig = go.Figure()
    for col in assets:
        fig.add_trace(go.Scatter(
            x=pct.index, y=pct[col],
            mode="lines", name=col,
            line=dict(color=COLOURS.get(col, "white"), width=2),
        ))

    fig.add_hline(y=0, line=dict(color="white", width=1, dash="dot"))

    # Optional event marker
    if event_date:
        fig.add_shape(
            type="line", x0=event_date, x1=event_date, y0=0, y1=1,
            xref="x", yref="paper",
            line=dict(color="white", width=2, dash="dash"),
        )
        if event_label:
            fig.add_annotation(
                x=event_date, y=1.02, xref="x", yref="paper",
                text=event_label, showarrow=False,
                font=dict(color="white", size=14),
            )

    update_timeseries_layout(fig)
    fig.update_layout(
        yaxis=dict(title=f"Return{title_suffix}", ticksuffix="%"),
        hovermode="x unified",
    )
    add_watermark(fig)
    return fig

In [40]:
# ══════════════════════════════════════════════════════════════════════
# Chart 1 — YTD normalised returns (since 1 Jan 2026)
# ══════════════════════════════════════════════════════════════════════
fig_ytd = normalised_return_chart(
    df_all, assets,
    start=YTD_START,
    title_suffix=" since 1 Jan 2026",
)
fig_ytd

In [10]:
# ══════════════════════════════════════════════════════════════════════
# Chart 2 — Returns since start of US–Iran conflict
# ══════════════════════════════════════════════════════════════════════
fig_conflict = normalised_return_chart(
    df_all, assets,
    start=CONFLICT_START,
    title_suffix=f" since US–Iran conflict",
)
fig_conflict

In [11]:
# ══════════════════════════════════════════════════════════════════════
# Chart 3 — Returns 13 Jun 2025 → 25 Jun 2025  (12-day war)
# Uses Yahoo Finance for S&P 500 (Bloomberg data doesn't go back this far)
# ══════════════════════════════════════════════════════════════════════
JUNE_START = "2025-06-13"
JUNE_END   = "2025-06-26"   # exclusive end for Yahoo; Bybit fetcher handles ms

# ── Bybit fetch for June window ──────────────────────────────────────
june_bybit = {}
for sym, label in BYBIT_SYMBOLS.items():
    print(f"Fetching {sym} ({JUNE_START} → {JUNE_END}) …", end=" ")
    kdf = fetch_bybit_klines(sym, interval=INTERVAL, start=JUNE_START, end=JUNE_END)
    kdf = kdf.rename(columns={"close": label})
    june_bybit[label] = kdf[label]
    print(f"{len(kdf)} rows")

df_june_bybit = pd.DataFrame(june_bybit).sort_index()

# ── Yahoo fetch for June window ──────────────────────────────────────
print(f"Fetching {YAHOO_SYMBOL} ({JUNE_START} → {JUNE_END}) …")
yf_june = yf.download(YAHOO_SYMBOL, start=JUNE_START, end=JUNE_END, interval="1h", auto_adjust=True)
yf_june.index = yf_june.index.tz_convert("UTC")
if isinstance(yf_june.columns, pd.MultiIndex):
    yf_june.columns = yf_june.columns.get_level_values(0)
df_june_yahoo = yf_june[["Close"]].rename(columns={"Close": YAHOO_LABEL})
df_june_yahoo_h = df_june_yahoo.resample("1h").last().dropna()
print(f"{len(df_june_yahoo_h)} rows")

# ── Merge & plot ─────────────────────────────────────────────────────
df_june = df_june_bybit.join(df_june_yahoo_h, how="left").sort_index()
df_june[YAHOO_LABEL] = df_june[YAHOO_LABEL].ffill()

june_assets = list(BYBIT_SYMBOLS.values()) + [YAHOO_LABEL]

fig_june = normalised_return_chart(
    df_june, june_assets,
    start=JUNE_START,
    title_suffix=" (13–25 Jun 2025)",
)
fig_june

Fetching BTCUSDT (2025-06-13 → 2025-06-26) … 313 rows
Fetching ETHUSDT (2025-06-13 → 2025-06-26) … 313 rows
Fetching XAUTUSDT (2025-06-13 → 2025-06-26) … 313 rows
Fetching ^GSPC (2025-06-13 → 2025-06-26) …


[*********************100%***********************]  1 of 1 completed

56 rows


In [12]:
# ══════════════════════════════════════════════════════════════════════
# Bloomberg-only: load all 4 assets from the Excel file
# ══════════════════════════════════════════════════════════════════════
_bbg_raw = pd.read_excel(BBG_SPX_FILE, sheet_name="Sheet1")
_bbg_raw["Date"] = (
    pd.to_datetime(_bbg_raw["Date"])
    .dt.tz_localize("US/Eastern", ambiguous="NaT", nonexistent="shift_forward")
    .dt.tz_convert("UTC")
)
_bbg_raw = _bbg_raw.dropna(subset=["Date"]).set_index("Date").sort_index()

df_bbg = _bbg_raw.rename(columns={
    "XBTUSD BGN Curncy  (L3)": "BTC",
    "XETUSD BGN Curncy  (L2)": "ETH",
    "XAU BGN Curncy  (R2)":    "XAU",
    "SPX Index  (L1)":         "SPX",
})[["BTC", "ETH", "XAU", "SPX"]]
df_bbg = df_bbg[~df_bbg.index.duplicated(keep="first")]
df_bbg["SPX"] = df_bbg["SPX"].ffill()

bbg_assets  = ["BTC", "XAU", "SPX", "ETH"]
BBG_COLOURS = {"BTC": "#ff7f0e", "ETH": "purple", "XAU": "#FFD700", "SPX": "#E63946"}

print(f"Bloomberg data: {len(df_bbg)} rows, {df_bbg.index.min()} → {df_bbg.index.max()}")
df_bbg[bbg_assets].dropna().tail(3)

Bloomberg data: 2199 rows, 2025-12-09 22:00:00+00:00 → 2026-03-11 12:00:00+00:00


,BTC,XAU,SPX,ETH
Date,,,,
2026-03-11 10:00:00+00:00,69612.38,5186.82,6781.48,2029.045
2026-03-11 11:00:00+00:00,69177.80,5180.74,6781.48,2018.145
2026-03-11 12:00:00+00:00,69345.94,5179.20,6781.48,2025.270


In [13]:
# ══════════════════════════════════════════════════════════════════════
# Bloomberg-only Chart 1 — YTD normalised returns (since 1 Jan 2026)
# ══════════════════════════════════════════════════════════════════════
fig_bbg_ytd = normalised_return_chart(
    df_bbg, bbg_assets,
    start=YTD_START,
    title_suffix=" since 1 Jan 2026",
)
for trace in fig_bbg_ytd.data:
    trace.line.color = BBG_COLOURS.get(trace.name, "white")
fig_bbg_ytd

In [14]:
# ══════════════════════════════════════════════════════════════════════
# Bloomberg-only Chart 2 — Returns since US–Iran conflict
# ══════════════════════════════════════════════════════════════════════
fig_bbg_conflict = normalised_return_chart(
    df_bbg, bbg_assets,
    start=CONFLICT_START,
    title_suffix=" since US–Iran conflict",
)
for trace in fig_bbg_conflict.data:
    trace.line.color = BBG_COLOURS.get(trace.name, "white")
fig_bbg_conflict

In [15]:
#### TAZMINA HELP 

In [16]:
# ══════════════════════════════════════════════════════════════════════
# BTC correlation setup — copy of macro-analysis template, adapted for SPX / Gold / Oil
# ══════════════════════════════════════════════════════════════════════
MACRO_FILE   = r"data\Macro-analysis-data.xlsx"
BTC_OIL_FILE = r"data\btc_oil_bybit_thahbib.xlsx"

_corr = pd.read_excel(MACRO_FILE, sheet_name="corr", parse_dates=["Date"]).set_index("Date")
_oil  = pd.read_excel(BTC_OIL_FILE, parse_dates=["Date"]).set_index("Date")

df_corr = pd.DataFrame({
    "BTC Spot": _corr["BTC Spot"],
    "SPX": _corr["SPX"],
    "XAU": _corr["XAU"],
    "Oil": _oil["CL1 COMB Comdty  (R1)"],
}).sort_index()

df_corr = df_corr.loc["2015-01-01":].copy()

COLOURS.setdefault("SPX", COLOURS.get("Bloomberg S&P 500", "red"))
COLOURS.setdefault("XAU", "#FFD700")
COLOURS.setdefault("OIL", "#3CB371")

print(f"Correlation source: {len(df_corr)} rows, {df_corr.index.min().date()} -> {df_corr.index.max().date()}")
display(df_corr)


Correlation source: 3826 rows, 2015-01-01 -> 2026-03-11


,BTC Spot,SPX,XAU,Oil
Date,,,,
2015-01-01,314.49,NaN,1181.98,NaN
2015-01-02,314.73,2058.20,1188.39,52.69
2015-01-04,266.26,NaN,NaN,NaN
2015-01-05,270.93,2020.58,1204.86,50.04
2015-01-06,286.41,2002.61,1218.58,47.93
...,...,...,...,...
2026-03-07,NaN,NaN,NaN,NaN
2026-03-08,NaN,NaN,NaN,NaN
2026-03-09,NaN,NaN,NaN,94.77


In [17]:
# ── Correlations from template: BTC vs SPX, Oil, XAU (30d) ───────────
correlation_window = 30
w = correlation_window

df_corr_30 = df_corr.copy()

# Returns
df_corr_30['BTC Daily Return'] = df_corr_30['BTC Spot'].pct_change()
df_corr_30['SPX Daily Return'] = df_corr_30['SPX'].pct_change()
df_corr_30['Oil Daily Return'] = df_corr_30['Oil'].pct_change()
df_corr_30['XAU Daily Return'] = df_corr_30['XAU'].pct_change()

# Drop rows with missing returns
df_corr_30 = df_corr_30.dropna(subset=[
    'BTC Daily Return', 'SPX Daily Return', 'Oil Daily Return', 'XAU Daily Return'
])

# Rolling correlations
df_corr_30[f'BTC_SPX_corr_{w}d'] = df_corr_30['BTC Daily Return'].rolling(window=w).corr(df_corr_30['SPX Daily Return'])
df_corr_30[f'BTC_OIL_corr_{w}d'] = df_corr_30['BTC Daily Return'].rolling(window=w).corr(df_corr_30['Oil Daily Return'])
df_corr_30[f'BTC_XAU_corr_{w}d'] = df_corr_30['BTC Daily Return'].rolling(window=w).corr(df_corr_30['XAU Daily Return'])

print(df_corr_30[[f'BTC_SPX_corr_{w}d', f'BTC_OIL_corr_{w}d', f'BTC_XAU_corr_{w}d']].dropna().tail(3))

# ── Chart: BTC rolling correlation vs SPX, Oil, Gold ─────────────────
w = correlation_window

fig_corr_30 = go.Figure()

for col, label, color in [
    (f'BTC_SPX_corr_{w}d', 'SPX', COLOURS['SPX']),
    (f'BTC_OIL_corr_{w}d', 'Oil', COLOURS['OIL']),
    (f'BTC_XAU_corr_{w}d', 'XAU', COLOURS['XAU']),
]:
    series = df_corr_30[col].dropna()
    fig_corr_30.add_trace(go.Scatter(
        x=series.index, y=series.values,
        mode='lines', name=label,
        line=dict(color=color, width=2),
    ))

fig_corr_30.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.4)

fig_corr_30.update_layout(
    showlegend=True, title="", height=650, width=1450,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="x unified",
    xaxis=dict(title="", automargin=True, tickangle=0,
               showgrid=True, gridwidth=1, gridcolor="#293e68",
               zeroline=False, showline=False, ticklen=10),
    yaxis=dict(title=f"BTC {w}d Rolling Correlation",
               gridcolor="#293e68", zeroline=False, showgrid=True),
    legend=dict(orientation="h", yanchor="top", y=1.1, xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=50, pad=4),
)
add_watermark(fig_corr_30)
display(fig_corr_30)


            BTC_SPX_corr_30d  BTC_OIL_corr_30d  BTC_XAU_corr_30d
Date                                                            
2026-03-09          0.494264          0.059146          0.263552
2026-03-10          0.494822          0.058110          0.267206
2026-03-11          0.511764          0.058282          0.281060


C:\Users\USER\AppData\Local\Temp\ipykernel_22604\1867998282.py:8: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22604\1867998282.py:9: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22604\1867998282.py:10: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22604\1867998282.py:11:

In [18]:
# ── Correlations from template: BTC vs SPX, Oil, XAU (90d) ───────────
correlation_window = 90
w = correlation_window

df_corr_90 = df_corr.copy()

# Returns
df_corr_90['BTC Daily Return'] = df_corr_90['BTC Spot'].pct_change()
df_corr_90['SPX Daily Return'] = df_corr_90['SPX'].pct_change()
df_corr_90['Oil Daily Return'] = df_corr_90['Oil'].pct_change()
df_corr_90['XAU Daily Return'] = df_corr_90['XAU'].pct_change()

# Drop rows with missing returns
df_corr_90 = df_corr_90.dropna(subset=[
    'BTC Daily Return', 'SPX Daily Return', 'Oil Daily Return', 'XAU Daily Return'
])

# Rolling correlations
df_corr_90[f'BTC_SPX_corr_{w}d'] = df_corr_90['BTC Daily Return'].rolling(window=w).corr(df_corr_90['SPX Daily Return'])
df_corr_90[f'BTC_OIL_corr_{w}d'] = df_corr_90['BTC Daily Return'].rolling(window=w).corr(df_corr_90['Oil Daily Return'])
df_corr_90[f'BTC_XAU_corr_{w}d'] = df_corr_90['BTC Daily Return'].rolling(window=w).corr(df_corr_90['XAU Daily Return'])

print(df_corr_90[[f'BTC_SPX_corr_{w}d', f'BTC_OIL_corr_{w}d', f'BTC_XAU_corr_{w}d']].dropna().tail(3))

# ── Chart: BTC rolling correlation vs SPX, Oil, Gold ─────────────────
w = correlation_window

fig_corr_90 = go.Figure()

for col, label, color in [
    (f'BTC_SPX_corr_{w}d', 'SPX', COLOURS['SPX']),
    (f'BTC_OIL_corr_{w}d', 'Oil', COLOURS['OIL']),
    (f'BTC_XAU_corr_{w}d', 'XAU', COLOURS['XAU']),
]:
    series = df_corr_90[col].dropna()
    fig_corr_90.add_trace(go.Scatter(
        x=series.index, y=series.values,
        mode='lines', name=label,
        line=dict(color=color, width=2),
    ))

fig_corr_90.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.4)

fig_corr_90.update_layout(
    showlegend=True, title="", height=650, width=1450,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="x unified",
    xaxis=dict(title="", automargin=True, tickangle=0,
               showgrid=True, gridwidth=1, gridcolor="#293e68",
               zeroline=False, showline=False, ticklen=10),
    yaxis=dict(title=f"BTC {w}d Rolling Correlation",
               gridcolor="#293e68", zeroline=False, showgrid=True),
    legend=dict(orientation="h", yanchor="top", y=1.1, xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=50, pad=4),
)
add_watermark(fig_corr_90)
display(fig_corr_90)


            BTC_SPX_corr_90d  BTC_OIL_corr_90d  BTC_XAU_corr_90d
Date                                                            
2026-03-09          0.552566          0.093135          0.222376
2026-03-10          0.557405          0.076950          0.222278
2026-03-11          0.556944          0.079663          0.221050


C:\Users\USER\AppData\Local\Temp\ipykernel_22604\303597768.py:8: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22604\303597768.py:9: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22604\303597768.py:10: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\USER\AppData\Local\Temp\ipykernel_22604\303597768.py:11: Fut

In [19]:
import statsmodels.api as sm
import plotly.express as px

In [20]:
# ── Chart 7b: BTC vs Oil scatter plot with OLS regression ─────────────
x_col      = "Oil"
y_col      = "BTC"
start_date = "2000-01-01"


oil_file = r"data\btc_oil_bybit_thahbib.xlsx"

oil_df = (
    pd.read_excel(oil_file, parse_dates=["Date"])
    .rename(columns={
        "CL1 COMB Comdty  (R1)": "Oil",
        "XBTUSD BGN Curncy  (L1)": "BTC",
    })
    .set_index("Date")
    .sort_index()
)

# Filter and prepare data
scatter_df = oil_df[[x_col, y_col]].loc[start_date:].dropna().copy()

# OLS regression
X = sm.add_constant(scatter_df[x_col])
model = sm.OLS(scatter_df[y_col], X).fit()
scatter_df["Regression_Line"] = model.predict(X)
#regression_label = f"R² = {model.rsquared:.4f}"

# Year labels for colour axis
scatter_df["Year"] = scatter_df.index.year.astype(str)
year_order = sorted(scatter_df["Year"].unique(), key=int)
year_to_num = {y: i for i, y in enumerate(year_order)}
scatter_df["YearNum"] = scatter_df["Year"].map(year_to_num)

# Scatter
fig7b = px.scatter(
    scatter_df,
    x=x_col, y=y_col,
    color="YearNum",
    color_continuous_scale="Peach",
    labels={"YearNum": "Year"},
)

# Regression line
"""fig7b.add_trace(go.Scatter(
    x=scatter_df[x_col],
    y=scatter_df["Regression_Line"],
    mode="lines", name=regression_label,
    line=dict(color="white", width=2),
))"""

# Highlight latest point
latest = scatter_df.iloc[-1]
fig7b.add_trace(go.Scatter(
    x=[latest[x_col]], y=[latest[y_col]],
    mode="markers", name="Latest",
    marker=dict(color="#14F195", size=15, symbol="x"),
    showlegend=True,
))

# Layout
fig7b.update_layout(
    height=1000, width=1000,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="closest", title="",
    xaxis=dict(title=x_col, showgrid=True, gridcolor="#293e68",
               zeroline=False, tickprefix="$",tickformat="~s"),
    yaxis=dict(title=y_col, tickprefix="$", showgrid=True,
               gridcolor="#293e68", zeroline=False),
    legend=dict(orientation="h", yanchor="top", y=1.1,
                xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=80, pad=4),
    coloraxis_colorbar=dict(
        title="",
        tickvals=list(year_to_num.values()),
        ticktext=list(year_to_num.keys()),
    ),
)
add_watermark(fig7b)
fig7b


In [21]:
# ── Chart 7b: BTC vs Oil scatter plot with OLS regression ─────────────
x_col      = "Oil"
y_col      = "BTC"
start_date = "2020-01-01"


oil_file = r"data\btc_oil_bybit_thahbib.xlsx"

oil_df = (
    pd.read_excel(oil_file, parse_dates=["Date"])
    .rename(columns={
        "CL1 COMB Comdty  (R1)": "Oil",
        "XBTUSD BGN Curncy  (L1)": "BTC",
    })
    .set_index("Date")
    .sort_index()
)

# Filter and prepare data
scatter_df = oil_df[[x_col, y_col]].loc[start_date:].dropna().copy()

# OLS regression
X = sm.add_constant(scatter_df[x_col])
model = sm.OLS(scatter_df[y_col], X).fit()
scatter_df["Regression_Line"] = model.predict(X)
regression_label = f"R² = {model.rsquared:.4f}"

# Year labels for colour axis
scatter_df["Year"] = scatter_df.index.year.astype(str)
year_order = sorted(scatter_df["Year"].unique(), key=int)
year_to_num = {y: i for i, y in enumerate(year_order)}
scatter_df["YearNum"] = scatter_df["Year"].map(year_to_num)

# Scatter
fig7b = px.scatter(
    scatter_df,
    x=x_col, y=y_col,
    color="YearNum",
    color_continuous_scale="Peach",
    labels={"YearNum": "Year"},
)

# Regression line
fig7b.add_trace(go.Scatter(
    x=scatter_df[x_col],
    y=scatter_df["Regression_Line"],
    mode="lines", name=regression_label,
    line=dict(color="white", width=2),
))

# Highlight latest point
latest = scatter_df.iloc[-1]
fig7b.add_trace(go.Scatter(
    x=[latest[x_col]], y=[latest[y_col]],
    mode="markers", name="Latest",
    marker=dict(color="#14F195", size=15, symbol="x"),
    showlegend=True,
))

# Layout
fig7b.update_layout(
    height=1000, width=1000,
    font=dict(size=20, color="white"),
    plot_bgcolor="#101A2E", paper_bgcolor="#101A2E",
    hovermode="closest", title="",
    xaxis=dict(title=x_col, showgrid=True, gridcolor="#293e68",
               zeroline=False, tickprefix="$",tickformat="~s"),
    yaxis=dict(title=y_col, tickprefix="$", showgrid=True,
               gridcolor="#293e68", zeroline=False),
    legend=dict(orientation="h", yanchor="top", y=1.1,
                xanchor="center", x=0.5),
    margin=dict(l=150, r=30, b=50, t=80, pad=4),
    coloraxis_colorbar=dict(
        title="",
        tickvals=list(year_to_num.values()),
        ticktext=list(year_to_num.keys()),
    ),
)
add_watermark(fig7b)
fig7b
